# Experimenting with MF-BPR and historical interactions 

historoical interactions are based on the categories of user's previously interacted locations. 

# MF-BPR model (after hyperparamter tuning)

In [45]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

In [46]:
# eval metrics 
import math

def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = len(set(recommended_k) & set(relevant))

    return hits / len(relevant)


def ndcg_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    # Actual DCG
    dcg = 0
    for i, item in enumerate(recommended_k):
        if item in relevant:
            dcg += 1 / math.log2(i + 2)

    # Best possible DCG
    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = set(recommended_k) & set(relevant)

    if len(hits) > 0:
        return 1
    else:
        return 0


def calculate_metrics(recommended, relevant, k=10):

    recall = recall_at_k(recommended, relevant, k)
    hit_rate = hit_rate_at_k(recommended, relevant, k)
    ndcg = ndcg_at_k(recommended, relevant, k)

    return {
        f"Recall@{k}": recall,
        f"HitRate@{k}": hit_rate,
        f"NDCG@{k}": ndcg
    }


In [47]:
# load training interactions 
train_df = pd.read_csv(
    "../data/processed/train60.csv",
    dtype={
        "user_id": "string",
        "gmap_id": "string",
        "rating" : float
    },
    usecols=["user_id", "gmap_id", "rating"]
)

train_df = train_df.drop_duplicates(
    ["user_id", "gmap_id"]
)

positive_train = train_df[
    train_df["rating"] >= 4
].copy()

In [48]:
#encode users and activities 
user_indices, unique_users = pd.factorize(
    train_df["user_id"],
    sort=True
)

activity_indices, unique_activities = pd.factorize(
    train_df["gmap_id"],
    sort=True
)

user_to_index = {
    user_id: index
    for index, user_id in enumerate(unique_users)
}

activity_to_index = {
    activity_id: index
    for index, activity_id in enumerate(unique_activities)
}

index_to_user = {
    index: user_id
    for index, user_id in enumerate(unique_users)
}

index_to_activity = {
    index: activity_id
    for index, activity_id in enumerate(unique_activities)
}

num_users = len(unique_users)
num_activities = len(unique_activities)

In [49]:
# Construct the sparse user–activity interaction matrix
user_activity_matrix = csr_matrix(
    (
        np.ones(len(train_df), dtype=np.float32),
        (user_indices, activity_indices)
    ),
    shape=(num_users, num_activities),
    dtype=np.float32
)

In [50]:
# Initialize the BPR model
bpr_model_60 = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.01,
    regularization=0.001,
    iterations=50,
    random_state=42
)

# Train using the sparse interaction matrix
bpr_model_60.fit(
    user_activity_matrix,
    show_progress=True
)

100%|██████████| 50/50 [03:08<00:00,  3.77s/it, train_auc=94.95%, skipped=0.15%]


In [51]:
def recommend_bpr(user_id, k=10):

    user_idx = user_to_index[user_id]

    recommended_items, scores = bpr_model_60.recommend(
        userid=user_idx,
        user_items=user_activity_matrix[user_idx],
        N=k,
        filter_already_liked_items=True
    )

    return recommended_items, scores

# Historical categories

In [52]:
import ast

activity_df = pd.read_csv(
    "../data/processed/activity_dataset.csv"
)

train_with_categories = positive_train.merge(
    activity_df[
        ["gmap_id", "name", "category", "activity_groups"]
    ],
    on="gmap_id",
    how="left"
)

train_with_categories["category"] = (
    train_with_categories["category"]
    .apply(
        lambda x: ast.literal_eval(x)
        if isinstance(x, str)
        else x
    )
)

train_with_categories["activity_groups"] = (
    train_with_categories["activity_groups"]
    .apply(
        lambda x: ast.literal_eval(x)
        if isinstance(x, str)
        else x
    )
)

place_metadata = (
    activity_df[
        ["gmap_id", "name", "category", "activity_groups"]
    ]
    .drop_duplicates("gmap_id")
    .copy()
)

for col in ["category", "activity_groups"]:
    place_metadata[col] = place_metadata[col].apply(
        lambda x: ast.literal_eval(x)
        if isinstance(x, str) and x.strip().startswith("[")
        else x
    )

In [53]:
def normalized_labels(value):
    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(label).strip()
            for label in value
            if str(label).strip()
        ]

    if isinstance(value, str):
        return [value.strip()] if value.strip() else []

    return []

In [54]:
# make category labels all lists 
def build_historical_preferences(column):

    history = train_with_categories[
        ["user_id", column]
    ].copy()

    history[column] = history[column].apply(
        normalized_labels
    )

    # Each interaction contributes a total weight of 1
    history["weight"] = (
        1.0 / history[column].str.len()
    )

    history = history.explode(column)

    history = history.dropna(
        subset=[column]
    )

    preference_counts = (
        history
        .groupby(["user_id", column])["weight"]
        .sum()
        .reset_index(name="preference_weight")
    )

    totals = (
        preference_counts
        .groupby("user_id")["preference_weight"]
        .transform("sum")
    )

    preference_counts["preference_score"] = (
        preference_counts["preference_weight"]
        / totals
    )

    return (
        preference_counts
        .groupby("user_id")
        .apply(
            lambda group: dict(
                zip(
                    group[column],
                    group["preference_score"]
                )
            )
        )
    )

In [55]:
historical_specific = (
    build_historical_preferences("category")
)

historical_broad = (
    build_historical_preferences("activity_groups")
)

In [56]:
def preference_reranker(
    user_id,
    candidates,
    bpr_weight=0.80,
    historical_weight=0.20
):
    candidates = candidates.copy()

    # Get this user's specific historical preferences
    try:
        specific_history = historical_specific.loc[user_id]
    except KeyError:
        specific_history = {}

    # Get this user's broad historical preferences
    try:
        broad_history = historical_broad.loc[user_id]
    except KeyError:
        broad_history = {}

    # Find the strongest matching historical preference
    def history_score(value, preferences):
        return max(
            (
                preferences.get(label, 0.0)
                for label in normalized_labels(value)
            ),
            default=0.0
        )

    # Specific category score
    candidates["historical_specific_score"] = (
        candidates["category"].apply(
            lambda value: history_score(
                value,
                specific_history
            )
        )
    )

    # Broad category score
    candidates["historical_broad_score"] = (
        candidates["activity_groups"].apply(
            lambda value: history_score(
                value,
                broad_history
            )
        )
    )

    # Normalize specific scores
    low = candidates["historical_specific_score"].min()
    high = candidates["historical_specific_score"].max()

    if high > low:
        candidates["historical_specific_normalized"] = (
            candidates["historical_specific_score"] - low
        ) / (high - low)
    else:
        candidates["historical_specific_normalized"] = 0.0

    # Normalize broad scores
    low = candidates["historical_broad_score"].min()
    high = candidates["historical_broad_score"].max()

    if high > low:
        candidates["historical_broad_normalized"] = (
            candidates["historical_broad_score"] - low
        ) / (high - low)
    else:
        candidates["historical_broad_normalized"] = 0.0

    # Split historical weight equally between specific and broad
    half_historical = historical_weight / 2

    candidates["rerank_score"] = (
        bpr_weight * candidates["bpr_normalized"]
        + half_historical
        * candidates["historical_specific_normalized"]
        + half_historical
        * candidates["historical_broad_normalized"]
    )

    candidates = candidates.sort_values(
        ["rerank_score", "bpr_rank"],
        ascending=[False, True]
    ).reset_index(drop=True)

    return candidates

In [57]:
def get_bpr_candidates(user_id, candidate_k=50):

    # Get BPR recommendations
    items, scores = recommend_bpr(
        user_id,
        k=candidate_k
    )

    candidates = pd.DataFrame({
        "gmap_id": [
            index_to_activity[i]
            for i in items
        ],
        "bpr_score": scores
    })

    # Keep original BPR ranking
    candidates["bpr_rank"] = (
        np.arange(len(candidates)) + 1
    )

    # Add activity metadata
    candidates = candidates.merge(
        place_metadata,
        on="gmap_id",
        how="left",
        validate="many_to_one"
    )

    # Normalize BPR scores
    min_score = candidates["bpr_score"].min()
    max_score = candidates["bpr_score"].max()

    if max_score > min_score:
        candidates["bpr_normalized"] = (
            (candidates["bpr_score"] - min_score)
            / (max_score - min_score)
        )
    else:
        candidates["bpr_normalized"] = 0.0

    return candidates

In [58]:
sample_df = pd.read_csv(
    "../data/processed/validation_sample_1000_users.csv",
    dtype={
        "user_id": "string",
    }
)

validation_df = pd.read_csv(
    "../data/processed/validation20.csv",
        dtype={
        "user_id": "string",
        "gmap_id": "string",
        "rating": float
    }
)

validation_pos = validation_df[
    validation_df["rating"] >= 4
].copy()

relevant_by_user = (
    validation_pos
    .groupby("user_id")["gmap_id"]
    .apply(set)
    .to_dict()
)

evaluation_users = sample_df[
    sample_df["user_id"].isin(relevant_by_user)
]["user_id"].tolist()

In [59]:
bpr_recall = []
bpr_hit_rate = []
bpr_ndcg = []

for user_id in evaluation_users:

    # Get BPR Top 10
    recommended_items, scores = recommend_bpr(
        user_id,
        k=10
    )

    # Convert internal item indices back to gmap_ids
    recommended = [
        index_to_activity[item_idx]
        for item_idx in recommended_items
    ]

    # Positive validation items for this user
    relevant = relevant_by_user[user_id]

    # Calculate metrics
    metrics = calculate_metrics(
        recommended,
        relevant,
        k=10
    )

    bpr_recall.append(metrics["Recall@10"])
    bpr_hit_rate.append(metrics["HitRate@10"])
    bpr_ndcg.append(metrics["NDCG@10"])


print("TUNED BPR")
print("Evaluated users:", len(evaluation_users))
print("Recall@10:", np.mean(bpr_recall))
print("Hit Rate@10:", np.mean(bpr_hit_rate))
print("NDCG@10:", np.mean(bpr_ndcg))

TUNED BPR
Evaluated users: 1000
Recall@10: 0.04173422082303661
Hit Rate@10: 0.097
NDCG@10: 0.025613993703137012


In [60]:
weight_configs = [
    (0.90, 0.10),
    (0.80, 0.20),
    (0.70, 0.30),
    (0.60, 0.40)
]

results = []

for bpr_weight, historical_weight in weight_configs:

    recall_scores = []
    hit_rate_scores = []
    ndcg_scores = []

    for user_id in evaluation_users:

        candidates = get_bpr_candidates(
            user_id,
            candidate_k=50
        )

        reranked = preference_reranker(
            user_id,
            candidates,
            bpr_weight=bpr_weight,
            historical_weight=historical_weight
        )

        recommended = (
            reranked
            .head(10)["gmap_id"]
            .tolist()
        )

        relevant = relevant_by_user[user_id]

        metrics = calculate_metrics(
            recommended,
            relevant,
            k=10
        )

        recall_scores.append(
            metrics["Recall@10"]
        )

        hit_rate_scores.append(
            metrics["HitRate@10"]
        )

        ndcg_scores.append(
            metrics["NDCG@10"]
        )

    results.append({
        "BPR Weight": bpr_weight,
        "Historical Weight": historical_weight,
        "Specific Weight": historical_weight / 2,
        "Broad Weight": historical_weight / 2,
        "Recall@10": np.mean(recall_scores),
        "HitRate@10": np.mean(hit_rate_scores),
        "NDCG@10": np.mean(ndcg_scores)
    })

    results_df = pd.DataFrame(results)

results_df

,BPR Weight,Historical Weight,Specific Weight,Broad Weight,Recall@10,HitRate@10,NDCG@10
0,0.9,0.1,0.05,0.05,0.042322,0.099,0.026499
1,0.8,0.2,0.10,0.10,0.041512,0.097,0.026411
2,0.7,0.3,0.15,0.15,0.038816,0.093,0.026011
3,0.6,0.4,0.20,0.20,0.038158,0.094,0.026185


The weighting of .9 bpr and .1 historical improved all 3 mertics slightly, compared to just the tuned mf-bpr.

In [61]:
weight_configs = [
    (0.98, 0.02),
    (0.95, 0.05),
    (0.92, 0.08),
    (0.90, 0.10),
    (0.85, 0.15),
    (0.80, 0.20)
]

results = []

for bpr_weight, historical_weight in weight_configs:

    recall_scores = []
    hit_rate_scores = []
    ndcg_scores = []

    for user_id in evaluation_users:

        candidates = get_bpr_candidates(
            user_id,
            candidate_k=50
        )

        reranked = preference_reranker(
            user_id,
            candidates,
            bpr_weight=bpr_weight,
            historical_weight=historical_weight
        )

        recommended = (
            reranked
            .head(10)["gmap_id"]
            .tolist()
        )

        relevant = relevant_by_user[user_id]

        metrics = calculate_metrics(
            recommended,
            relevant,
            k=10
        )

        recall_scores.append(
            metrics["Recall@10"]
        )

        hit_rate_scores.append(
            metrics["HitRate@10"]
        )

        ndcg_scores.append(
            metrics["NDCG@10"]
        )

    results.append({
        "BPR Weight": bpr_weight,
        "Historical Weight": historical_weight,
        "Specific Weight": historical_weight / 2,
        "Broad Weight": historical_weight / 2,
        "Recall@10": np.mean(recall_scores),
        "HitRate@10": np.mean(hit_rate_scores),
        "NDCG@10": np.mean(ndcg_scores)
    })

    results_df = pd.DataFrame(results)

results_df

,BPR Weight,Historical Weight,Specific Weight,Broad Weight,Recall@10,HitRate@10,NDCG@10
0,0.98,0.02,0.010,0.010,0.042131,0.100,0.025893
1,0.95,0.05,0.025,0.025,0.042409,0.101,0.026064
2,0.92,0.08,0.040,0.040,0.042155,0.100,0.026135
3,0.90,0.10,0.050,0.050,0.042322,0.099,0.026499
4,0.85,0.15,0.075,0.075,0.042112,0.096,0.026447
5,0.80,0.20,0.100,0.100,0.041512,0.097,0.026411
